# Legislative Content Extraction Agent - Google ADK

Notebook for the `LegislativeContentExtractionAgent` that extracts structured
legislative metadata from US legislative PDF documents using Google ADK + Gemini.

**Requirements:** `GOOGLE_API_KEY` (or `GEMINI_API_KEY`) environment variable must be set.

In [ ]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# Load .env from project root and add it to sys.path
PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "legislative_content_extraction" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")
sys.path.insert(0, str(PROJECT_ROOT))

api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
assert api_key, "GOOGLE_API_KEY or GEMINI_API_KEY not set. Add it to your .env file."

IMPL_DIR = (Path.cwd()) if Path.cwd().name == "legislative_content_extraction" else (Path.cwd() / "implementations" / "legislative_content_extraction")
FILES_DIR = IMPL_DIR / "files"
DATASET_JSON = IMPL_DIR / "legislative_content_extraction_dataset.json"

# Load dataset
dataset = json.load(open(DATASET_JSON)) if DATASET_JSON.exists() else []
record_ids = [r["record_id"] for r in dataset if r.get("record_id")]
print(f"Found {len(record_ids)} record(s): {record_ids}")

def get_record(record_id: str) -> dict:
    for record in dataset:
        if record.get("record_id") == record_id:
            return record
    return {}

## 1. Test the `read_pdf` tool standalone

In [ ]:
from aieng.agent_evals.legislative_content_extraction.tools import read_pdf

record = get_record(record_ids[1])
record_id = record["record_id"]
content_dir = FILES_DIR / record_id
pdf_name = record.get("pdf_file_name", f"{record_id}.pdf")
PDF_PATH = str(content_dir / pdf_name)

result = read_pdf(PDF_PATH)
print(f"Status: {result['status']}")
print(f"Pages: {result['num_pages']}")
print(f"Content preview:\n{result['content'][:500]}")

## 2. Run the agent manually

In [ ]:
from aieng.agent_evals.legislative_content_extraction import LegislativeContentExtractionAgent

agent = LegislativeContentExtractionAgent(files_dir=str(content_dir))

In [ ]:
html_page_link = record.get("html_page_link", "")
print(f"HTML page link: {html_page_link or '(none)'}")

response = await agent.answer_async(
    pdf_path=PDF_PATH,
    prompt="Extract legislative metadata from this bill.",
    html_page_link=html_page_link,
)

print(f"Duration: {response.total_duration_ms}ms")
print(f"Tool calls: {response.tool_calls}")

## 3. Inspect output

In [ ]:
print(response.text)

In [ ]:
print("Reasoning chain:")
for i, step in enumerate(response.reasoning_chain, 1):
    print(f"  {i}. {step}")

print(f"\nToken usage: {agent.token_tracker.usage}")   

---
## 4. Interactive Gradio App

Run the cell below to launch an interactive UI for selecting PDFs and extracting metadata.

In [ ]:
from implementations.legislative_content_extraction.demo import start_gradio_app

start_gradio_app(enable_public_link=True)